# SponsorBlock-AI — ModernBERT Training Notebook

**Run on Google Colab (GPU runtime) or Kaggle (GPU/TPU).**

Steps:
1. Install deps
2. Download SponsorBlock DB
3. Fetch transcripts + build windows dataset
4. Fine-tune ModernBERT-base
5. Evaluate
6. Push to HuggingFace Hub

Set `HF_TOKEN` in Colab Secrets (key icon in sidebar) before step 6.

In [ ]:
# 1. Install
!pip install -q 'sponsorblock-ai[train] @ git+https://github.com/chirag127/sponsorblock-ai.git'
# Or from local mount:
# !pip install -q '/content/sponsorblock-ai[train]'

In [ ]:
import os, logging
logging.basicConfig(level=logging.INFO)

# Colab secret → env var
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass  # Kaggle: set via Kaggle Secrets → environment variables

In [ ]:
# 2. Download SponsorBlock DB
from pathlib import Path
from sponsorblock_ai.data.fetch_db import fetch_sponsor_times
from sponsorblock_ai.constants import CATEGORIES

sb_df = fetch_sponsor_times(
    dest_dir=Path('data/raw'),
    min_votes=2,
    categories=CATEGORIES,
)
print(sb_df['category'].value_counts())
print(f'Total rows: {len(sb_df):,}')

In [ ]:
# 3. Build windows dataset
# Sample video IDs from the DB (adjust max_videos for faster iteration)
from sponsorblock_ai.data.build_dataset import build_windows_dataframe, build_hf_dataset

video_ids = sb_df['videoID'].drop_duplicates().tolist()
print(f'{len(video_ids):,} unique videos in DB')

# Start with 500 videos; increase for production training
windows_df = build_windows_dataframe(
    sb_df,
    video_ids[:500],
    window_size=128,
    stride=64,
)
print(windows_df['label'].value_counts())
windows_df.to_parquet('data/windows.parquet', index=False)

In [ ]:
# Or load cached parquet if already built
import pandas as pd
windows_df = pd.read_parquet('data/windows.parquet')
print(f'{len(windows_df):,} windows')

In [ ]:
# Tokenise + split
dataset_dict = build_hf_dataset(windows_df)
print(dataset_dict)

In [ ]:
# 4. Fine-tune
from sponsorblock_ai.model.trainer import train

trainer, test_results = train(
    dataset_dict,
    output_dir='outputs/sponsorblock-modernbert',
    num_epochs=3,
    batch_size=32,
    learning_rate=2e-5,
    fp16=True,           # set False on TPU
    push_to_hub=False,   # set True after verifying results
    hub_model_id='chirag127/sponsorblock-modernbert',
)

print('Test results:', test_results)

In [ ]:
# 5. Confusion matrix
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sponsorblock_ai.constants import CATEGORIES

pred_output = trainer.predict(dataset_dict['test'])
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=CATEGORIES)
fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45)
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# 6. Push to Hub (requires HF_TOKEN)
# Re-run train() with push_to_hub=True, or push manually:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = 'outputs/sponsorblock-modernbert'
hf_id = 'chirag127/sponsorblock-modernbert'
token = os.environ.get('HF_TOKEN')

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained('answerdotai/ModernBERT-base')

model.push_to_hub(hf_id, token=token)
tokenizer.push_to_hub(hf_id, token=token)
print(f'Pushed to https://huggingface.co/{hf_id}')